In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import gradio as gr

# ==========================================
# 1. EXACT MODEL ARCHITECTURE (MUST MATCH NOTEBOOK)
# ==========================================
class MultimodalNeuralCrypto(nn.Module):
    def __init__(self, vocab_size=128, text_seq_len=16, embed_dim=64, key_dim=16):
        super(MultimodalNeuralCrypto, self).__init__()
        self.text_seq_len = text_seq_len
        self.embed_dim = embed_dim

        # ALICE (ENCRYPTION / ENCODER)
        self.img_encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, embed_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )
        self.text_embedding = nn.Embedding(vocab_size, embed_dim)
        self.text_rnn_encoder = nn.GRU(embed_dim, embed_dim, batch_first=True)
        self.key_projector = nn.Linear(key_dim, embed_dim)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=128, batch_first=True
        )
        self.transformer_scrambler = nn.TransformerEncoder(enc_layer, num_layers=2)

        # BOB (DECRYPTION / DECODER)
        dec_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=128, batch_first=True
        )
        self.transformer_descrambler = nn.TransformerEncoder(dec_layer, num_layers=2)

        self.text_rnn_decoder = nn.GRU(embed_dim, embed_dim, batch_first=True)
        self.text_out = nn.Linear(embed_dim, vocab_size)

        self.img_decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def encrypt(self, img, text_tokens, secret_key):
        b = img.size(0)
        img_feats = self.img_encoder(img)
        img_tokens = img_feats.view(b, self.embed_dim, -1).permute(0, 2, 1)

        key_emb = self.key_projector(secret_key).unsqueeze(0)
        txt_emb = self.text_embedding(text_tokens)
        txt_tokens, _ = self.text_rnn_encoder(txt_emb, key_emb)

        combined = torch.cat([img_tokens, txt_tokens, key_emb.permute(1, 0, 2)], dim=1)
        return self.transformer_scrambler(combined)

    def decrypt(self, cipher_tensor, secret_key):
        b = cipher_tensor.size(0)
        key_emb = self.key_projector(secret_key).unsqueeze(1)
        unlocked = cipher_tensor + key_emb

        descrambled = self.transformer_descrambler(unlocked)
        img_tokens = descrambled[:, :16, :]
        txt_tokens = descrambled[:, 16:16 + self.text_seq_len, :]

        img_feats = img_tokens.permute(0, 2, 1).view(b, self.embed_dim, 4, 4)
        rec_img = self.img_decoder(img_feats)

        rnn_out, _ = self.text_rnn_decoder(txt_tokens, key_emb.permute(1, 0, 2))
        rec_text_logits = self.text_out(rnn_out)
        return rec_img, rec_text_logits


# ==========================================
# 2. LOAD PRE-TRAINED .PTH WEIGHTS AT STARTUP
# ==========================================
MODEL_PATH = "multimodal_neural_crypto.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultimodalNeuralCrypto().to(device)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Could not find '{MODEL_PATH}'. Download it from Colab and place it in the same folder as app.py."
    )

# map_location ensures GPU-trained weights load cleanly on CPU machines
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval() # Lock model into inference mode
print(f"Successfully loaded '{MODEL_PATH}' on {device}!")


# ==========================================
# 3. PRE/POST PROCESSING HELPERS
# ==========================================
def string_to_key_tensor(key_str):
    seed = sum(ord(c) * (i + 1) for i, c in enumerate(key_str)) % (2**32 - 1)
    rng = np.random.default_rng(seed)
    key_vec = rng.standard_normal((1, 16)).astype(np.float32)
    return torch.tensor(key_vec, device=device)

def text_to_tensor(text_str, seq_len=16):
    if not text_str:
        text_str = "A" * seq_len
    cleaned = text_str.upper().replace(" ", "_")
    padded = cleaned.ljust(seq_len, "_")[:seq_len]
    ascii_ids = [min(ord(c), 127) for c in padded]
    return torch.tensor([ascii_ids], dtype=torch.long, device=device)

def tensor_to_text(logits):
    pred_ids = logits.argmax(dim=-1).squeeze(0).cpu().tolist()
    return "".join(chr(max(32, min(i, 126))) for i in pred_ids)

def image_to_tensor(pil_img):
    if pil_img is None:
        pil_img = Image.fromarray(np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8))
    img_resized = pil_img.convert("RGB").resize((32, 32))
    arr = np.array(img_resized, dtype=np.float32) / 255.0
    return torch.tensor(arr).permute(2, 0, 1).unsqueeze(0).to(device)

def tensor_to_image(img_tensor):
    arr = img_tensor.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(arr).resize((256, 256), Image.NEAREST)

def cipher_to_visual_image(cipher_tensor):
    flat = cipher_tensor.detach().cpu().numpy().flatten()
    norm = (flat - flat.min()) / (flat.max() - flat.min() + 1e-8)
    grid = (norm[:2048].reshape(32, 64, 1) * 255).astype(np.uint8)
    rgb_noise = np.repeat(grid, 3, axis=2)
    return Image.fromarray(rgb_noise).resize((256, 256), Image.NEAREST)


# ==========================================
# 4. INSTANT ENCRYPTION & DECRYPTION CALLBACKS
# ==========================================
def run_encryption(pil_img, secret_msg, enc_key_str):
    img_t = image_to_tensor(pil_img)
    txt_t = text_to_tensor(secret_msg)
    key_t = string_to_key_tensor(enc_key_str)

    with torch.no_grad():
        cipher_tensor = model.encrypt(img_t, txt_t, key_t)
        cipher_img = cipher_to_visual_image(cipher_tensor)

    return cipher_tensor, cipher_img, "Encrypted instantly using pre-trained weights."

def run_decryption(cipher_tensor, dec_key_str):
    if cipher_tensor is None:
        return None, "ERROR: Encrypt an image and message first."
    key_t = string_to_key_tensor(dec_key_str)

    with torch.no_grad():
        rec_img_t, rec_txt_logits = model.decrypt(cipher_tensor, key_t)
        out_img = tensor_to_image(rec_img_t)
        out_txt = tensor_to_text(rec_txt_logits)

    return out_img, out_txt


# ==========================================
# 5. GRADIO WEB INTERFACE
# ==========================================
with gr.Blocks(title="Instant Multimodal Neural Cryptography") as demo:
    gr.Markdown("## 🔐 Multimodal Neural Cryptography (Pre-Trained Inference)")
    gr.Markdown(f"**Loaded Checkpoint:** `{MODEL_PATH}` | **Device:** `{device}`")

    cipher_state = gr.State(None)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Alice (Encrypt)")
            in_img = gr.Image(type="pil", label="Confidential Image (Auto-resized to 64x64)")
            in_txt = gr.Textbox(
                label="Confidential Message (Max 16 ASCII chars)",
                value="TOP_SECRET_12345",
                interactive=True
            )
            enc_key = gr.Textbox(label="Encryption Secret Key", value="my_secret_key_99", type="password")
            enc_btn = gr.Button("🔒 Encrypt Instantly", variant="primary")

        with gr.Column():
            gr.Markdown("### 2. Scrambled Cipher-Tensor")
            cipher_vis = gr.Image(label="Transformer Latent Noise Heatmap", interactive=False)
            status_box = gr.Textbox(label="Status", interactive=False)

        with gr.Column():
            gr.Markdown("### 3. Bob (Decrypt)")
            dec_key = gr.Textbox(label="Decryption Key (Edit 1 char to test wrong key)", value="my_secret_key_99")
            dec_btn = gr.Button("🔓 Decrypt", variant="secondary")
            out_img = gr.Image(label="Reconstructed Image", interactive=False)
            out_txt = gr.Textbox(label="Reconstructed Text", interactive=False)

    enc_btn.click(
        fn=run_encryption,
        inputs=[in_img, in_txt, enc_key],
        outputs=[cipher_state, cipher_vis, status_box]
    )
    dec_btn.click(
        fn=run_decryption,
        inputs=[cipher_state, dec_key],
        outputs=[out_img, out_txt]
    )

if __name__ == "__main__":
    demo.launch()

Successfully loaded 'multimodal_neural_crypto.pth' on cpu!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a831f97c442c6bdf6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
